# A Deep Agent is just a regular agent with
- A small collection of essential tools
- A small collection of prebuilt middleware

### a few superpowers
- The ability to read from and write to filesystems
- The ability to spawn (sync/async) subagents

### and the ability to plan
- The builtin **todo list** middleware

In [1]:
%load_ext autoreload
%autoreload 2

In [13]:
from util import pretty_print, print_todos, print_exchange, print_activity
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI
import os
from langchain.agents.middleware import TodoListMiddleware

from dotenv import load_dotenv
load_dotenv(override=True)

# model = "claude-haiku-4-5-20251001"

model = ChatOpenAI(
    openai_api_key= os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    # model_name="gpt-5.5",
    model_name="gpt-5.6-luna",
    # Deep Agents uses function tools; reasoning + tools requires /v1/responses.
    use_responses_api=True,
    
)
    
research_system_prompt = """You are an expert research assistant."""

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    # middleware=[TodoListMiddleware()]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What tools do you have?"}]})
pretty_print(result)

### Human

What tools do you have?

### Ai

I can:

- **Search and inspect files** in the available workspace:
  - List directories
  - Find files by glob patterns
  - Search file contents for literal text
  - Read files, including text and some media/PDF content
- **Edit the workspace**:
  - Make precise text replacements
  - Create or fully replace files
  - Delete files or directories
- **Delegate complex tasks** to a temporary general-purpose subagent for independent research or multi-step work.

I don’t have general internet browsing or terminal/shell access unless those capabilities are exposed through the workspace tools.

# The TODO list in action

#### Ask a deep agent to perform a complex task and it starts by planning out all the steps.

In [5]:
result = agent.invoke({"messages": [{"role": "user", "content": "I need to bake a cake. Make a plan. Use TodoList Middleware"}]})
print_todos(result)

*No todos.*

# Tools

#### Deep agents, as well as their subagents, can be extended with tools. Let's make ours more useful by allowing it to search the internet.

In [27]:
@tool
def search(query: str) -> str:
    """Search the internet for information

    Args:
        query (str): search query
    """
    results = TavilyClient().search(query)
    return results.get("results", [])

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    tools=[search]
)

result = agent.invoke({"messages": [{"role": "user", "content": "What are the top 2 stories in the news today?"}]})
pretty_print(result)

### Human

What are the top 2 stories in the news today?

### Tool

[{"url": "https://apnews.com", "title": "Associated Press News: Breaking News, Latest Headlines ...", "content": "The U.S. Navy aircraft carrier USS Abraham Lincoln arrives at the port in Laem Chabang, Chonburi, Thailand, Wednesday, Sept. 2, 2026. (AP Photo/Sakchai Lalit)\n\n### USS Abraham Lincoln crew visits Thailand’s Pattaya, in photos\n\n[deltaMinutes] mins ago Now\n\n 15\n\nNepal police personnel prepare the bodies of flash flood victims for a mass burial in Kathmandu, Nepal, Tuesday, Sept. 1, 2026. (AP Photo/Niranjan Shrestha)\n\n### The top photos of the day by AP’s photojournalists [...] Why Hormuz matters\n\nA man sits with a child injured, according to Iranian media, in a strike on a home where a wedding was being held in Kuhestak, southern Iran, at a hospital in Menab, Iran, early Wednesday, Sept.2, 2026. (Razieh Poudat/ISNA via AP)\n\nA man sits with a child injured, according to Iranian media, in a strike on a home where a wedding was being held in Kuhestak, southern Iran, at a hospital in Menab, Iran, early Wednesday, Sept.2, 2026. (Razieh Poudat/ISNA via AP) [...] SECTIONS\n\n  2026 Elections Election Results Election calendar White House Congress Supreme Court The latest AP-NORC polls\n\n  TOP STORIES\n\n  + US military completes latest strikes against Iran, while Trump vows more for Tehran's retaliation\n  + Justice Department seizes more than $560,000 in cryptocurrency donations for Hamas\n  + Bessent says 19 finance ministers agreed 'cheap exports' are unsustainable, but China dissented\n\n  Newsletters\n\n  Ground Game", "score": 0.6562404, "raw_content": null, "id": "6103ee-00"}, {"url": "https://www.npr.org/sections/news", "title": "U.S. and World News Headlines : NPR", "content": "Los Angeles Clippers owner Steve Ballmer was suspended for one year by the NBA, and the team was ordered to pay $30 million for violating salary cap circumvention rules in a case involving Kawhi Leonard.\n\nRussia’s increased attacks on Kyiv and its partnership with Iran\n\nFirefighters put out a fire at warehouse stores after a Russian air attack in Kyiv, Ukraine, Wednesday, Sept. 2, 2026.  Efrem Lukatsky/AP  hide caption\n\n toggle caption\n\n Efrem Lukatsky/AP\n\n### State of the World from NPR [...] Police forensic technicians work near a Ukrainian aircraft at Leipzig/Halle Airport, where a drone with an explosive device was found and a detonator was removed from the object, in Schkeuditz, Germany, Wednesday Aug. 5, 2026.  Hendrik Schmidt/DPA via AP  hide caption\n\n toggle caption\n\n Hendrik Schmidt/DPA via AP\n\n### Europe\n\n## Germany blames Russia for last month's attempted drone attack at Leipzig airport [...] If the high court accepts the petition, it would tee up a case with sweeping implications for prediction markets, an industry that has unleashed billions of dollars of betting across nearly every aspect of modern life.\n\nPeople wait outside the Miami Immigration Court on Wednesday, Sept. 2, 2026 in Miami, Fla. Eva Marie Uzcategui for NPR", "score": 0.6469293, "raw_content": null, "id": "6bae09-01"}, {"url": "https://www.reuters.com/world/us", "title": "Latest U.S. News | Top headlines from the USA - Reuters", "content": "Cookies, opens new tab\n   Terms & Conditions\n   Privacy, opens new tab\n   Copyright, opens new tab\n   Digital Accessibility, opens new tab\n   Corrections\n   Data Disclosure and Sources, opens new tab\n   Site Feedback, opens new tab\n   Commercial Disclosure (Japan), opens new tab\n   Manage Cookies\n\n© 2026 Reuters. All rights reserved\n\nReport AdImage 19\n\nImage 20Image 21Image 22Image 23Image 24Image 25Image 26Image 27Image 28Image 29Image 30Image 31Image 37: dot image pixel", "score": 0.46935064, "raw_content": null, "id": "650e6c-02"}, {"url": "https://www.ap.org", "title": "AP.org", "content": "Spotlights\n\n### Photos show revelers celebrating Caribbean culture at London’s Notting Hill Carnival\n\nAUG. 31, 2026\n\nWeekend revelers attended J’ouvert, the opening event of the Notting Hill Carnival, before crowds packed the streets of west London on Monday for the climax of one of the world’s biggest celebrations of Caribbean music and culture.\n\nRead more [...] AUG. 31, 2026\n\nSome of Hollywood’s biggest stars are heading to Venice, Italy, for the 83rd edition of the Venice Film Festival, which kicks off on Wednesday. Here’s a rundown of everything you need to know about the festival, the Oscar buzz and who’s going. The Venice Film Festival banks on star power It’s one of the most prestigious film […]\n\nRead more]( [...] AUG. 31, 2026\n\nA plan in one of Texas’ largest counties to use dozens fewer polling locations than in previous years is drawing protests from Democrats who say the reductions could create confusion in the November midterm elections.\n\nRead more\n\n[This photo provided by NASA shows technicians connecting the inner and outer segments of NASA’s Nancy Grace Roman Space Telescope, Nov. 25, 2025, at NASA’s Goddard Space Flight Center in Greenbelt, Md. (Jolearra Tshiteya/NASA via AP)\n\nSpotlights", "score": 0.4560954, "raw_content": null, "id": "fe2ef4-03"}, {"url": "https://www.cnn.com", "title": "CNN: Breaking News, Latest News and Videos", "content": "Nathan Howard/Reuters Kennedy Center removes outdoor ‘Blue’ sculpture\n From Pierre Damas Bel An Ohio college student died in an apparent suicide. His family says he felt ‘humiliated’ by his ICE ankle monitor\n [  \n\n  CNN](/2026/09/01/us/tiger-woods-dui-plea-change-hearing) Tiger Woods avoids DUI conviction but loses driving privileges for 5 years\n [ [...] • Video  2:06\n\n  CNN Video Woman accused of filming Clancy jurors pleads not guilty 2:06\n • Video  0:45\n\n  Pool via Video Tiger Woods reaches plea deal in DUI arrest 0:45\n • Video  0:27\n\n  WUSA Video Kennedy Center has dismantled ‘Blue’ sculpture 0:27\n • Video  1:12\n\n  KCBS/KCAL Video Angels fans finally get the news they’ve been waiting for 1:12\n • Video  2:16\n\n  CNN Video Data center backlash is rocking 2026 midterms 2:16\n • Video  3:00 [...] Video 3:01\n\n  Clipped From Video Video ‘Heart-wrenching’: Nepali police take CNN inside search for flood victims 3:01\n Clipped From Video Race against time to find workers missing in Nepal’s hydropower tunnels. Here’s what we know\n\nAd Feedback\n\nAd Feedback\n\n## CNN Underscored\n\n Joe Bloss/CNN Underscored Abercrombie, The North Face and other clothing brands just launched Labor Day deals\n\n  Sep 2, 2026\n Michelle Rae Uy/CNN Underscored The best padlocks in 2026, tried and tested", "score": 0.36983892, "raw_content": null, "id": "88ef43-04"}]

### Tool

[{"url": "https://www.bbc.com/news", "title": "BBC News - Breaking news, video and the latest top stories from the U.S. and around the world", "content": "Centre Benhard Janse van Rensburg admits he \"probably would have loved to\" play for his native South Africa, but denies Rassie Erasmus' claims he was in talks over a Springboks call-up.\n\nA collage of images and illustrations\n\n## Sign up to News Briefing\n\n### The BBC is in multiple languages\n\n#### Read the BBC In your own language\n\nCopyright 2026 BBC. All rights reserved. The BBC is not responsible for the content of external sites. Read about our approach to external linking. [...] It comes amid concerns over the growing number of traders from neighbouring countries.\n\n## MoD withheld information about nuclear test veterans' records, ex-minister claims\n\nCold War veterans who oversaw nuclear tests have claimed they caused lifelong health problems.\n\nThe Prime Minister, Pedro Sanchez, addresses a plenary session of the Congress of Deputies on 3 September 2026 in Madrid\n\n## Embattled Spanish PM insists there were no prior warnings before Ceuta crisis [...] European Commission President Ursula von der Leyen (R)and NATO Secretary General Mark Rutte arrive for a press statement at EU headquarters on September 2, 2026\n\n## EU and Nato vow to step up pressure on Russia after 'new escalation' in Germany\n\nRussia is accused of growing increasingly reckless after Berlin blamed Russia for a failed drone attack on Leipzig airport.\n\n## Most watched\n\n## US man struck by lightning describes 'horrific' experience", "score": 0.7159213, "raw_content": null, "id": "68f8a3-00"}, {"url": "https://www.bbc.com", "title": "BBC Home - Breaking News, World News, US News, Sports ...", "content": "The firm's Hong Kong listing on Tuesday comes after a years-long quest to sell shares in New York and London.\n\nAn elderly woman walks past a destroyed vehicle following an Israeli strike in the Rimal neighbourhood of Gaza City (1 September 2026)\n\n## Israeli fire kills three in Gaza City during reported raid, medics say\n\nUnconfirmed local reports say the Israeli military carried out air and artillery strikes after a raid by an Israel-backed militia was discovered. [...] Women looking at a screen to identify loved ones who have died at a mortuary in Nepal\n\n## Nepal death toll crosses 1,000 as rescuers race to find workers trapped in mud-filled tunnels\n\nThousands are still missing as Chinese state media report the official number of fatalities in Tibet at 16.\n\nCCTV footage shows people fleeing as a wall of mud and water destroys buildings at the Nepal-China border in Gyirong, Tibet, China, on August 26, 2026. [...] A wooden building with two cannons on the grass in front, and mist in front of a woodland hillside behind (Credit: Alamy)\n\n## 'The forgotten empire': When Russia colonised California\n\nFor nearly 30 years, part of Sonoma was ruled by Tsarist Russia, and it played a pivotal – if little-known – role in shaping the US.\n\n## Football\n\nMalick Fofana and Ainsley Maitland-Niles\n\n## Transfer Deadline Day 2026 - Palace could hijack Fofana deal, Everton eye Maitland-Niles", "score": 0.55362654, "raw_content": null, "id": "822c50-01"}, {"url": "https://www.bbc.com/news/uk", "title": "UK | Latest News & Updates - BBC", "content": "## Laundry rooms to be turned into cells to tackle prison overcrowding\n\nThe government says £110m has been fast-tracked from existing budgets to speed up the building of new cells.\n\nA wide shot of SAN PAWL. It is a large white vessel with a blue stripe running along the side. It is docking in a harbour.\n\n## Ferry service remains suspended after engine fault\n\nIslands Unlimited says services between Jersey and Guernsey will remain paused until 10 September. [...] Copyright 2026 BBC. All rights reserved. The BBC is not responsible for the content of external sites. Read about our approach to external linking. [...] ## 'Orange Order put us through hell' say Garvaghy Road residents\n\nBreandán Mac Cionnaith said the Orange Order should be apologising to the people rather than marching in the area.\n\nA woman with long brown hair and a yellow top sitting in a restaurant. She is laughing and is covering her mouth with her hand.\n\n## Family of murdered Stacy Hunter say new misogyny bill would not have protected her\n\nFirst Minister John Swinney said he wanted to use his role to tackle violence against women.", "score": 0.50915486, "raw_content": null, "id": "19ee20-02"}, {"url": "https://www.bbc.com/news/world", "title": "World | Latest News & Updates", "content": "## US Supreme Court clears way for Trump to keep building White House ballroom\n\nBut Chief Justice John Roberts dissents, writing that \"construction is likely unlawful\".\n\n## Features\n\n## What it was like inside court for Tupac Shakur’s murder trial verdict\n\nBBC presenter Anoushka Mutanda-Dougherty describes what it was like when Duane \"Keffe D\" Davis was found guilty of murdering the famous rapper.\n\n## Flock's rapidly expanding AI surveillance network facing growing backlash in US [...] BBC Verify examines the expanding network of Flock cameras in the US and the backlash against it.\n\n## Japan Inc is betting big on India as China risks deepen\n\nJapanese companies are doubling down on India amid a shrinking domestic market and rising risks in China.\n\n## The jury's options in Lindsay Clancy's murder trial\n\nJurors have five options before them as they decide the fate of the former nurse charged with killing her three children. [...] 1 hr ago\nAsia\n\n## South Korean police officer admits to closing missing person cases to avoid work\n\nThe officer falsely claimed he contacted missing persons and prematurely closed their cases on the South Korean holiday island of Jeju.\n\n3 hrs ago\nAsia\n\n## Shein shares fall in long-awaited stock market debut\n\nThe firm's Hong Kong listing on Tuesday comes after a years-long quest to sell shares in New York and London.\n\n5 hrs ago\nBusiness", "score": 0.34059536, "raw_content": null, "id": "d46e94-03"}, {"url": "https://www.bbc.com/audio/play/p0nwy8c5", "title": "Global News Podcast | US July 4th celebrations disrupted", "content": "Get in touch: globalpodcast@bbc.co.uk\n\nPhoto: Spectators at Great American State Fair in Washington celebrating 250th anniversary of US independence.\n\nCredit: Graeme Sloan/EPA/Shuttercock\n\nSee more episodes\n\nProgramme website\n\n### The BBC is in multiple languages\n\n#### Read the BBC In your own language\n\nCopyright 2026 BBC. All rights reserved. The BBC is not responsible for the content of external sites. Read about our approach to external linking. [...] Global News Podcast, Jeffrey Epstein, Howard Lutnick, and the disgraced former prince\n\nJeffrey Epstein, Howard Lutnick, and the disgraced former prince\n\nGlobal News Podcast, Spain shatter Argentina's World Cup dream\n\nSpain shatter Argentina's World Cup dream\n\nGlobal News Podcast, Burnham becomes UK's seventh PM in a decade\n\nBurnham becomes UK's seventh PM in a decade\n\n5 July 2026\n\nAvailable for 34 days [...] Bangkok bar fire kills at least 27\n\nGlobal News Podcast, Trials of new Ebola vaccine set to begin\n\nTrials of new Ebola vaccine set to begin\n\nGlobal News Podcast, US reinstates naval blockade on Iran\n\nUS reinstates naval blockade on Iran\n\nGlobal News Podcast, ICE officer kills a Colombian man in Maine\n\nICE officer kills a Colombian man in Maine\n\nGlobal News Podcast, Trump scraps threat of 20% fee on Hormuz cargo\n\nTrump scraps threat of 20% fee on Hormuz cargo", "score": 0.17766848, "raw_content": null, "id": "ed4c02-04"}]

### Tool

[{"url": "https://www.cnn.com/world", "title": "World news - breaking news, video, headlines and opinion | CNN", "content": "In this pool photograph distributed by the Russian state agency Sputnik, Russia's President Vladimir Putin meets with the media after the Shanghai Cooperation Organisation (SCO) summit in Bishkek on September 1, 2026. (Photo by Vyacheslav PROKOFYEV / POOL / AFP via Getty Images)\nclipped thumbnail -  - CNN ID 22941292 - 00:00:03;15\nJapan is rolling out everything from robotic wolves to AI and drones to manage a rising bear problem [...] TEL AVIV, ISRAEL - AUGUST 24: Israeli Minister of National Security, Itamar Ben Gvir, speaks during a press conference at the Etzel Museum following the discovery of the Altalena ship after 78 years at bottom of the Mediterranean sea on August 24, 2026 in Tel Aviv, Israel. The wreckage of the ship Altalena has been found fully intact at a depth of 503 metres off the coast of Israel. She was a decommissioned US Navy ship purchased by the Irgun paramilitary group that arrived during Israel's War [...] Liberia's national police officers stand guard for a plane carrying migrants deported from the US at the Roberts International Airport in Harbel, some 56 km outside Liberia's capital, Monrovia, on August 20, 2026.", "score": 0.6407488, "raw_content": null, "id": "3d3a43-00"}, {"url": "https://www.cnn.com", "title": "CNN: Breaking News, Latest News and Videos", "content": "• Video  2:06\n\n  CNN Video Woman accused of filming Clancy jurors pleads not guilty 2:06\n • Video  0:45\n\n  Pool via Video Tiger Woods reaches plea deal in DUI arrest 0:45\n • Video  0:27\n\n  WUSA Video Kennedy Center has dismantled ‘Blue’ sculpture 0:27\n • Video  1:12\n\n  KCBS/KCAL Video Angels fans finally get the news they’ve been waiting for 1:12\n • Video  2:16\n\n  CNN Video Data center backlash is rocking 2026 midterms 2:16\n • Video  3:00 [...] US \n  + Crime & Justice\n  + Immigration\n  + Education\n  + Transportation\n  + Race & Identity\n World \n  + Africa\n  + Americas\n  + Asia\n  + Australia\n  + China\n  + Europe\n  + India\n  + Middle East\n  + United Kingdom\n Politics \n  + Elections 2026\n  + Trump\n  + Facts First\n  + CNN Polls\n  + Redistricting Tracker\n  + Epstein Files\n Business \n  + Tech\n  + Media\n  + Calculators\n  + Videos\n Markets \n  + Pre-markets\n  + After-Hours\n  + Fear & Greed\n  + Investing\n Health \n  + Life, But Better [...] US \n  + Crime & Justice\n  + Immigration\n  + Education\n  + Transportation\n  + Race & Identity\n World \n  + Africa\n  + Americas\n  + Asia\n  + Australia\n  + China\n  + Europe\n  + India\n  + Middle East\n  + United Kingdom\n Politics \n  + Elections 2026\n  + Trump\n  + Facts First\n  + CNN Polls\n  + Redistricting Tracker\n  + Epstein Files\n Business \n  + Tech\n  + Media\n  + Calculators\n  + Videos\n Markets \n  + Pre-markets\n  + After-Hours\n  + Fear & Greed\n  + Investing\n Health \n  + Life, But Better", "score": 0.6271529, "raw_content": null, "id": "61ee59-01"}, {"url": "https://www.cnn.com/business", "title": "Business News - Latest Headlines on CNN Business", "content": "## War in Iran\n\nVessels near the Strait of Hormuz, as seen from Musandam, Oman, August 17, 2026.\nAn oil products tanker is anchored in the Strait of Hormuz on August 10, 2026 off the coast of Bandar Abbas, Iran.\nA sign displays the price of unleaded gasoline at an Exxon gas station in Washington, DC, on May 21, 2026.\nThe Lutine Bell hangs in the center of the underwriting room at Lloyd's of London. The bell was recovered in 1858 from the wreck of the HMS Lutine, which sank in 1799. [...] Artist rendering of the Lunar Outpost Mobile Autonomous Prospecting Platform (MAPP) rover with Nokia antennas extended. The images, data and telemetry collected by MAPP will be sent back to the Nova-C lander over the Nokia 4G/LTE network and then relayed to Earth. Mission controllers in Colorado will issue commands to the rover over the same connection. [...] © 2026 Cable News Network. A Warner Bros. Discovery Company. All Rights Reserved.   \n CNN Sans ™ & © 2016 Cable News Network.\n\n## Download the CNN app\n\nScan the QR code to download the CNN app on Google Play.\n\nScan the QR code to download the CNN app from Google Play.\n\n## Download the CNN app\n\nScan the QR code to download the CNN app from the Apple Store.\n\nScan the QR code to download the CNN app from the Apple Store.", "score": 0.62330866, "raw_content": null, "id": "8ee00e-02"}, {"url": "https://www.cnn.com/us", "title": "US - CNN", "content": "after a woman was arrested for video recording jurors leaving the courthouse.  Wednesday,September 2, 2026\r\rPool Photo by Greg Derr/The Patriot Ledger [...] after a woman was arrested for video recording jurors leaving the courthouse.  Wednesday,September 2, 2026\r\rPool Photo by Greg Derr/The Patriot Ledger [...] Airport staff and officials attend to a plane carrying migrants deported from the US at the Roberts International Airport in Harbel, some 56 km outside of Liberia's capital of Monrovia, on August 20.\nBuildings are seen in a flood damaged Bright Angel Canyon the Grand Canyon, Monday, Aug. 31, 2026, in Grand Canyon National Park, Ariz. (AP Photo/John Locher)\nPolice respond to a shooting Wednesday, September 2, 2026, in downtown Minneapolis. (", "score": 0.48616135, "raw_content": null, "id": "121d1c-03"}]

### Tool

[{"url": "https://en.wikipedia.org/wiki/2026_Iran_war", "title": "2026 Iran war", "content": "656. ↑ \"Trump vows response to Iranian strike on US troops in Jordan\". The Times of Israel. 31 August 2026.\n657. ↑ \"Iranian media say Saudi oil tanker stopped in Strait of Hormuz, no Saudi confirmation\". Haaretz. Reuters. 31 August 2026.\n658. ↑ \"Tanker struck by three projectiles while sailing out of Strait of Hormuz, monitor says\". The Times of Israel. Reuters. 1 September 2026. [...] On 1 September, the British military announced that a tanker was hit by three projectiles while getting out of the Strait of Hormuz. The US announced that it struck IRGC targets in Iran, adding that it targeted tankers owned by the National Iranian Oil Company as part of a new \"tanker for tanker\" policy. Trump said that one target was Iranian work to rebuild radar systems. Iranian state media said that US targets included Jiroft Airport, and that a US strike in the vicinity of Bandar Sirik [...] 559. ↑ \"Trump Says Pilots Safe After US Army Apache Goes Down Near Hormuz\". Reuters. 9 June 2026.\n560. ↑ \"Drone Boat Rescues Crew of Downed U.S. Apache Helicopter Near Hormuz\". The Wall Street Journal. 9 June 2026.\n561. 1 2 \"US Military Launches New Strikes on Iran After Apache Downing\". Reuters. 9 June 2026.\n562. ↑ \"Trump Says Iran Shot Down US Helicopter over Hormuz, Vows to Respond\". Al Jazeera English. 9 June 2026.", "score": 0.6530611, "raw_content": null, "id": "98a258-00"}, {"url": "https://www.reuters.com/world/trump-says-latest-strikes-iran-are-justified-warns-tehran-against-retaliation-2026-09-01", "title": "Trump says latest strikes on Iran are 'justified,' warns Tehran against ...", "content": "All quotes delayed a minimum of 15 minutes. See here for a list of exchanges and delays.\n\n© 2026 Reuters. All rights reserved [...] Iran releases purported footage of missile launches at US targets\nIran releases footage purporting to show missiles launched at US military targets in Jordan\nA drone view shows vessels near the Strait of Hormuz, as seen from Musandam\n\nSign up  here.\n\nReporting ⁠by ​Bhargav Acharya, ​Susan Heavey and Ismail Shakil;\nEditing ​by David Ljunggren\n\nOur Standards: The Thomson Reuters Trust Principles., opens new tab\n\n## Read Next [...] ## Browse World\n\n## Browse Business\n\n## Browse Markets\n\n## Browse Sustainability\n\n## Legal\n\n## Commentary\n\n## Technology\n\n## Investigations\n\n## Sports\n\n## Science\n\n## Lifestyle\n\n## City Memo\n\n## Graphics\n\n## Pictures\n\n## Wider Image\n\n## Podcasts\n\n## Live\n\n## Fact Check\n\n## Video\n\n## Media Center\n\n## Sponsored Content\n\n# Trump says latest strikes on Iran are 'justified,' warns Tehran against retaliation", "score": 0.4600226, "raw_content": null, "id": "1b842a-01"}, {"url": "https://www.reuters.com/world/middle-east/us-military-says-it-completed-latest-wave-strikes-iran-2026-09-02", "title": "US military says it completed latest wave of strikes on Iran", "content": "All quotes delayed a minimum of 15 minutes. See here for a list of exchanges and delays.\n\n© 2026 Reuters. All rights reserved [...] Reuters, the news and media division of Thomson Reuters, is the world’s largest multimedia news provider, reaching billions of people worldwide every day. Reuters provides business, financial, national and international news to professionals via desktop terminals, the world's media organizations, industry events and directly to consumers.\n\n### Follow Us\n\n### LSEG Products [...] USS Abraham Lincoln sails toward Laem Chabang Port in Chonburi province, Thailand\nPanama Canal to cap daily transits, anticipating severe El Niño\nAlba summit, in Caracas\nU.S. Senate Armed Services Committee hearing regarding the Defense Authorization Request for Fiscal Year 2027 and the Future Years Defense Program, in Washington\n\n### Site Index\n\n### Latest\n\n### Media\n\n### Latest\n\n### Browse\n\n### Media\n\n### About Reuters\n\n### Stay Informed\n\n### Information you can trust", "score": 0.4051433, "raw_content": null, "id": "3729f9-02"}, {"url": "https://www.cnbc.com/2026/09/01/us-strikes-iran-after-new-hormuz-strait-shipping-attacks-centcom.html", "title": "U.S. strikes Iran as Tehran retaliates, raising risk of wider war", "content": "Livestream\n\n Make It\n select\n\n USA\n INTL\n\n Livestream\n\n Livestream\n\nWatchlist\n\nSIGN IN\n\nCreate free account\n\nMarkets\n\nBusiness\n\nInvesting\n\nTech\n\nPolitics & Policy\n\nVideo\n\nWatchlist\n\nInvesting Club\n\nJoin IC\n\nPRO\n\nJoin Pro\n\nLivestream\n\nPolitics & Policy\n\n# U.S. strikes Iran after new Hormuz Strait shipping attacks: Centcom\n\nWATCH LIVE\n\nThis is breaking news. Please refresh for updates. [...] Choose CNBC as your preferred source on Google and never miss a moment from the most trusted name in business news. [...] Investing\n\n Personal Finance\n Fintech\n Financial Advisors\n Options Action\n ETF Street\n Buffett Archive\n Earnings\n Trader Talk\n\nTech\n\n Cybersecurity\n AI\n Enterprise\n Internet\n Media\n Mobile\n Social Media\n CNBC Disruptor 50\n Tech Guide\n\nPolitics & Policy\n\n White House\n Policy\n Defense\n Congress\n Expanding Opportunity\n Europe Politics\n China Politics\n Asia Politics\n World Politics\n\nVideo", "score": 0.39901572, "raw_content": null, "id": "ddf48f-03"}, {"url": "https://www.reuters.com/world/middle-east/iran-urges-us-comply-with-interim-deal-after-trump-threatens-further-strikes-2026-09-01", "title": "Iran urges US to comply with interim deal after Trump ...", "content": "All quotes delayed a minimum of 15 minutes. See here for a list of exchanges and delays.\n\n© 2026 Reuters. All rights reserved [...] ## Read Next\n\nA currency dealer holds one-hundred-dollar bills, as the value of the Iranian rial drops, in Tehran\nPalestinians gather at the site of Israeli strikes and gunfire in Gaza City\nA drone view shows vessels near the Strait of Hormuz, as seen from Musandam\nScenes from the Port of Fujairah, as U.S.-Israel conflict with Iran limits marine traffic in the Strait of Hormuz\nVessels near the Strait of Hormuz, as seen from Musandam\nLSEG Workspace\n\n## World\n\nReuters logo [...] Billboard depicting U.S. President Donald Trump, on a building in Tehran\nVessels near the Strait of Hormuz, as seen from Musandam\nShanghai Cooperation Organisation summit in Bishkek\n\nSign up  here.\n\n## FEARS THAT IRAN CONFLICT COULD 'RUN AND RUN'\n\n## ECONOMIC PRESSURE", "score": 0.35716435, "raw_content": null, "id": "73ca77-04"}]

### Tool

[{"url": "https://www.youtube.com/watch?v=Ied4ovhug1c", "title": "Nepal death toll crosses 1000 as rescuers find workers ...", "content": "### Description\n26670 views\nPosted: 1 Sep 2026\nThe death toll in Nepal from devastating floods on the border with Tibet has crossed 1,000, according to the country's disaster agency.\n \nMore than 3,900 people remain missing - of whom more than 600 are workers believed to be trapped in a vast network of mud-filled tunnels at hydropower sites along Trishuli River. [...] The number of confirmed dead in Nepal and Tibet from last week's flash floods has passed 1,000. More than 3,900 people remain missing, of whom more than 600 are workers believed to be trapped in a vast network of mudfilled tunnels at hydropower sites along the Trishuli River. Engineers have been pumping air into the tunnels and while a team did manage to gain entry to one of them, it withdrew after finding no one there. Our South Asia correspondent Azad Masheri has been to Dun that is one of [...] # Nepal death toll crosses 1,000 as rescuers find workers trapped in mud-filled tunnels | BBC News\n## BBC News\n20000000 subscribers\n534 likes", "score": 0.8965509, "raw_content": null, "id": "fb8017-00"}, {"url": "https://www.thehindu.com/news/international/nepal-flash-floods-live-updates-rescue-operations-relief-death-toll-missing-persons-news-september-2-2026/article71417912.ece", "title": "Nepal flash floods aftermath LIVE: Death toll rises to ... - The Hindu", "content": "Follow LIVE updates here\n\n##### Death toll crosses 1,000 as rescuers recover more bodies; nearly 4,000 still missing\n\nThe death toll from the devastating flash floods in Nepal surged to 1,050 on Tuesday (September 1, 2026), with security forces recovering more bodies from eight districts of the country, officials said.\n\nRescuers struggled to trace those stranded amid intermittent rainfall and a fresh flood alert as water levels rose in rivers across eastern, central and western parts of Nepal. [...] The death toll from the devastating flash floods in Nepal surged to 1,050 on Tuesday (September 1, 2026), with security forces recovering more bodies from eight districts of the country, officials said. Nearly 4,000 people, including 583 foreigners and 83 security personnel, still remain missing the National Disaster Risk Reduction and Management Authority (NDRRMA) said.\n\nWatch | How a school in Nepal saved 900 students from a devastating flash flood [...] India   World   Sport   Data   Health   Opinion   Science   Entertainment   Premium\n\nVideos   Podcast   Photos   Visual Stories   Specials   TH Explains\n\neBooks   TH Games   Newsletter   Lit For Life   The Huddle\n\nThe Hindu Logo\n\nLive\n\n# Nepal flash floods aftermath LIVE: Death toll crosses 1,000 as rescuers recover more bodies; nearly 4,000 still missing\n\n## Nepal's Prime Minister Balendra Shah said that the devastating flood that entered Rasuwa was due to climate change", "score": 0.8732259, "raw_content": null, "id": "17d5d6-01"}, {"url": "https://www.theguardian.com/world/live/2026/sep/01/nepal-tibet-floods-death-toll-rescue-hydropower-tunnels-china-latest-updates", "title": "Nepal-Tibet floods: millions at risk from glacial ...", "content": "Skip to navigationSkip to key events\n\nPrint subscriptions\n\nSearch jobs\n\nSign in\n\nNepal\n\n# Nepal-Tibet floods: death toll passes 1,000 as rescue efforts continue at hydropower tunnels – latest updates\n\nAbout 4,500 people are missing across Nepal and Tibet after the disaster, including 900 feared trapped in mud-filled tunnels at 12 Nepali hydropower sites\n\nLIVE Updated\n\nYohannes Lowe\n\nTue 1 Sep 2026 09.56 CESTFirst published on Tue 1 Sep 2026 09.54 CEST\n\nShare\n\nKey events [...] Two ​Nepali officials said China had ​not shared much information on glacier risks and water levels after a meeting to strengthen cooperation earlier this year, and they feared that could hamper future disaster preparedness.\n\nlocator\n\nYou can read more in this story by my colleagues, Cait KellyandWilliam Christou, here:\n\nNepal rescuers race to reach hundreds feared trapped in hydropower tunnels\n\nRead more\n\n09.54 CEST\n\n## Death toll from Nepal-China flood disaster exceeds 1,000, authorities say [...] Many people are feared to still be trapped in the sprawling network of mud-filled tunnels at hydropower sites along the Trishuli River, which flows through central Nepal.\n\nKey events\n\n Death toll from Nepal-China flood disaster exceeds 1,000, authorities say\n\n09.55 CEST\n\nThe Red Cross estimates that more than 93,000 people have been affected by the disaster, heightening fears that the climate crisis is destabilising the geology of mountain and polar regions.", "score": 0.8708723, "raw_content": null, "id": "6f9a84-02"}, {"url": "https://www.youtube.com/watch?v=9tJFIE39YA0", "title": "Nepal flood death toll crosses 1000, 80 Americans remain ...", "content": "# Nepal flood death toll crosses 1,000, 80 Americans remain missing\n## News 4 (WOAI) San Antonio\n171000 subscribers\n154 likes\n\n### Description\n76935 views\nPosted: 2 Sep 2026\nThe death toll from catastrophic flash flooding along the Nepal-China border has surpassed 1,000 as rescue teams search through mud, rocks and debris for thousands of people still missing. [...] toll in China and Nepal following the devastating flash flood continues to rise with more than a thousand confirmed dead and four times that still missing. Reed Binyon has the latest. Rescuers in Nepal working in a hydropower tunnel where scores of workers are feared to be trapped after last week's devastating flash floods. Officials say around 900 workers at multiple hydropower projects are unaccounted for. Getting stuck in a tunnel, you can't see anything. You generally feel If you stay in [...] Nepalese authorities have begun burying unidentified bodies in temporary graves as they preserve DNA samples, photograph distinguishing features and record other details to help families identify and claim the remains later.\n\nRescuers are also racing to reach workers believed trapped in damaged hydropower tunnels. About 900 workers are missing from 12 projects, and local authorities believe roughly 500 may be trapped underground.", "score": 0.8184036, "raw_content": null, "id": "e25c49-03"}, {"url": "https://www.bbc.com/news/articles/c5ylvgq9q71o", "title": "Nepal death toll crosses 1000 as rescuers race to find ...", "content": "It comes as the death toll from the flood disaster on the Nepal-Tibet border reached 939, according to the country's disaster management authority.\n\nThe agency also revised its estimate of the number of missing to 3,925, of whom 592 are foreign nationals.\n\nRoughly one in seven missing people are believed to be workers at the hydropower plants. Nepalese Prime Minister Balendra Shah said on Monday that 279 people had been rescued from the sites. [...] The rescue operation is being spearheaded by an international team made up of Chinese and Nepalese military personnel, while experts from India, Korea and Australia are also involved.\n\nReuters A rescuer works at the entrance of a tunnel at Chilime Hydropower Plant as rescue operations continue for people trapped inside following deadly flash floods and a mudslide, in Rasuwa, Nepal. [...] # Nepal rescuers blast hillside in search of hydropower workers as families wait anxiously\n\nNepal Army/Reuters People in camouflage clothing and wearing helmets stand at the entry to a tunnel\n\nThe families of more than 600 workers believed to be trapped in a vast network of mud-filled tunnels along Nepal's Trishuli River are clinging to hope their loved ones can be found alive.", "score": 0.7206637, "raw_content": null, "id": "bb211c-04"}]

### Tool

[{"url": "https://foreignpolicy.com/2026/09/03/russia-failed-drone-attack-germany-leipzig-airport-eu-nato", "title": "The Fallout of Russia’s Failed Drone Attack on Germany", "content": "Europe and NATO “won’t just maintain our pressure on Russia. We will increase it,” European Commission President Ursula von der Leyen wrote on X on Wednesday, adding to a previous message pledging the EU’s “full solidarity” with Germany. Meanwhile, NATO chief Mark Rutte called the alliance’s commitment to collective defense “ironclad” and promised to “continue to strengthen our ability to deter and defend all Allies against any threat—including malign actions like those seen in Leipzig.” [...] Follow FP on X\n\nFollow FP on LinkedIn\n\nFollow FP on Instagram\n\nFollow FP on Facebook\n\nForeign Policy Magazine is a division of Graham Holdings Company. All contents (c) 2026, Foreign Policy LLC, a unit of Graham Digital Holding Company. All rights reserved. Foreign Policy, 1099 14th St NW, Suite 500 East, Washington, D.C., 20005.\n\n# The Fallout of Russia’s Failed Drone Attack on Germany\n\n## European nations join Berlin in punishing Moscow for an alleged unsuccessful attack on Leipzig airport. [...] Username\n\nI agree to abide by FP’s comment guidelines. (Required)\n\nLoading...\n\n## Editors’ Picks\n\nThis article has an audio recording\nThis article has an audio recording\n\n## Latest\n\n### EU, NATO Punish Russia for an Alleged Drone Attack on Germany's Leipzig Airport\n\n### The United States Must Renew Its Pushback Against Russia\n\n### China Shrugs Off Trump's Iran Economy Threats\n\n### Turkey’s Kurdish Peace Process Is Not About Erdogan", "score": 0.7036111, "raw_content": null, "id": "335cea-00"}, {"url": "https://www.youtube.com/watch?v=KNJVsuD9LJY", "title": "Pressure On Russia To Be Amped Up After Leipzig Airport Drone Attack", "content": "# Pressure On Russia To Be Amped Up After Leipzig Airport Drone Attack: EU & NATO Leaders | FULL\n## Forbes Breaking News\n5600000 subscribers\n1728 likes\n\n### Description\n290339 views\nPosted: 2 Sep 2026\nEU Commission President Ursula von der Leyen and NATO Sec.-Gen. Mark Rutte hold a press briefing in Brussels, Belgium\n\nStay Connected\nForbes Breaking News on X: \nForbes Breaking News on TikTok: \nMore From Forbes: [...] Union soil. We will not tolerate it. We stand in full solidarity with Germany. And if the intention was to make us think twice about supporting Ukraine, I can tell you it has only strengthened my resolve. Intimidation will not work, and this is why I'm meeting with NATO's Secretary General today. We are united. Europe and NATO will not just maintain our pressure on Russia, we will increase it, and we will not stop. Not until Russia stops bombing and killing innocent children, men, and women in [...] treat our everyday infrastructure as frontline targets and protect them accordingly. And we will support any member state Russia targets. Europe has already rapid response teams, which can respond to these style of attacks, whether they occur inside or outside our Union. So, our strength, our preparedness, and our resolve is our deterrence. We also do this by using the power of our single market. Our sanctions have weakened Russia's economy. We can more use more sanctions and have them ready to", "score": 0.70263255, "raw_content": null, "id": "aa6f9f-01"}, {"url": "https://www.aljazeera.com/news/2026/9/1/germany-accuses-russia-of-drone-attack-on-leipzig-airport", "title": "Germany accuses Russia of failed drone attack on Leipzig airport in August | News | Al Jazeera", "content": "Live\n\nLive\n\nBREAKINGBREAKING,\n\nNews\n\n# Germany accuses Russia of drone attack on Leipzig airport\n\nSave\n\nShare\n\nBreaking News logo for Al Jazeera\n\n(Al Jazeera)\n\nPublished On 1 Sep 20261 Sep 2026\n\nThe German government has accused Russia of being behind an attempted drone attack on a major regional airport.", "score": 0.67140645, "raw_content": null, "id": "81ed97-02"}, {"url": "https://www.bbc.com/news/articles/c5ylm3m67n2o", "title": "Germany says Russia behind Leipzig airport drone attack", "content": "It failed to explode as the detonator was apparently faulty, and investigators suspect a second drone collided with a cargo aircraft nearby. A third drone was reportedly found near the airport 10 days later.\n\nInterior Minister Alexander Dobrindt said the Leipzig airport attack followed the \"well-established pattern of Russian hybrid operations in Europe\" and the assumption was that Germany would be targeted again. [...] She added that the EU was preparing additional sanctions against Russia.\n\nMeanwhile, British Prime Minister Andy Burnham said: \"Russian attempts to undermine our united resolve and our willingness to support Ukraine are futile and doomed to fail.\"\n\nGermany is one of several Nato member states to have reported drone sightings at airports, military and industrial sites over the last year. [...] The Russian foreign minister said it would respond to \"anti-Russian decisions\", Interfax news agency reported. Security experts had speculated that Russia may have been responsible, however Russia has always rejected claims of involvement.\n\nLeipzig/Halle Airport in eastern Germany is used by the Germany military and Nato allies for the transport of military goods and also serves as a base for Ukraine's Antonov Airlines - which carries most of the country's aerial cargo.", "score": 0.6501347, "raw_content": null, "id": "14b730-03"}, {"url": "https://www.npr.org/2026/09/02/g-s1-141502/germany-russia-attempted-drone-attack", "title": "Germany blames Russia for attempted drone attack at Leipzig airport", "content": "Leipzig/Halle is a major hub for international freight and for providing support to Ukraine. The planes that regularly use it include Ukrainian Antonov transport aircraft.\n\nIt is also home to NATO's Strategic Airlift International Solution, or SALIS, which is currently operational almost every day in delivering equipment to NATO battle groups along its eastern flank from Finland to Romania and supporting EU and alliance nation missions. [...] Germany will propose the addition of more people from Russia to European Union sanctions lists \"in the context of hybrid attacks,\" step up entry controls for people from Russia and increase \"pressure on the shadow fleet,\" a reference to tankers suspected of shipping Russian oil in violation of international sanctions, Wadephul said. The EU has sanctioned hundreds of \"shadow fleet\" ships.\n\nThe Russian ambassador was summoned to the foreign ministry in Berlin. [...] A forensic technician works alongside a bomb-disposal robot near a Ukrainian aircraft at Leipzig/Halle Airport, where a drone with an explosive device was found and a detonator was removed from the object, in Schkeuditz, Germany, Wednesday Aug. 5, 2026.  Hendrik Schmidt/DPA via AP  hide caption\n\n toggle caption\n\n Hendrik Schmidt/DPA via AP", "score": 0.61363083, "raw_content": null, "id": "4c268d-04"}]

### Ai

As of today, the two biggest global news stories appear to be:

1. **U.S.–Iran conflict escalates after latest U.S. strikes**
   - AP and Reuters are leading with U.S. military strikes against Iran, with President Trump warning Tehran against further retaliation.
   - The story is especially significant because of risks to the Strait of Hormuz, global oil/shipping routes, and wider regional conflict.

2. **Nepal–Tibet flood disaster passes 1,000 deaths**
   - BBC, The Guardian, and other outlets report the death toll from catastrophic flash floods has crossed 1,000, with thousands still missing.
   - Rescue teams are racing to reach workers believed trapped in mud-filled hydropower tunnels along Nepal’s Trishuli River.

News rankings vary by outlet and region, but these are among the most prominent and consequential stories across major international sources today.

# Task Delegation: Where Deep Agents Shine

#### The typical pattern is one main supervisor agent delegating to one or more specialized subagents.

### Each subagent has:
- its own context window
- its own tools
- its own system prompt
- its own model (can override the main agent's model)
- its own skills

### The main agent
- delegates via its task() tool
- sees only the final results of its subagents, keeping its context window clean

![Deep agent task delegation](images/deep-agent-delegation.svg)

# Defining Subagents: Dictionary Approach

#### Just define your subagent as a Python **dictionary** and plug it into your supervisor main agent.

In [28]:
research_subagent = {
    "name": "research-agent",
    "description": "Research a given task",
    "system_prompt": f"""You are a research assistant.
                         Use tools to gather information.
                         Structure findings with clear headings and inline citations.
                         Limit to 3 search calls.""",
    "model": "anthropic:claude-haiku-4-5-20251001",
    "tools": [search]
}

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[research_subagent])

# Defining Subagents: Compiled Approach

For more complex workflows we can use **compiled** subagents, allowing us arbitrary complexity.

In [30]:
from langchain.agents import create_agent
from deepagents import CompiledSubAgent

research_subagent = create_agent(
    model=model,
    tools=[search],
    system_prompt="""You are a research assistant.
                     Use the search tool to gather information.
                     Structure findings with clear headings and inline citations.
                     Limit to 3 search calls.""",
)

compiled_subagent = CompiledSubAgent(
    name="research-agent", description="Research a given task using web search", runnable=research_subagent,
)

agent = create_deep_agent(model=model, system_prompt=research_system_prompt, subagents=[compiled_subagent])

# Filesystems and backends

#### Deep agents are designed to handle long-running, complex tasks. They typically do this by both offloading excess context that accumulates during execution and decomposing tasks into sub-tasks and delegating execution to subagents. Deep agents expose a virtual **filesystem** surface to allow (sub)agents to read and write state through a collection of tools: `ls, read_file, write_file, edit_file, glob, and grep`. The actual backend type is pluggable and abstracted from the agent.

A few prebuilt filesystem backends that you can quickly use with your deep agent:

<table style="font-size:18px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:12px; text-align:left;">Backend</th>
<th style="border:1px solid #999; padding:12px; text-align:left;">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Default<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash")</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped. The default filesystem backend for an agent is stored in langgraph state. Files persist across turns within a thread (via your checkpointer) and are not shared across threads.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local filesystem persistence<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=FilesystemBackend(root_dir="/Users/nh/Desktop/"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the deep agent access to your local machine's filesystem. You can specify the root directory that the agent has access to. Any provided <code style="font-size:15px; color:#2b8a3e;">root_dir</code> must be an absolute path. Typically, wrap in a <code style="font-size:15px; color:#2b8a3e;">CompositeBackend</code> to keep internal agent data (offloaded tool results, conversation history) separate from your project files.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Durable store (LangGraph store)<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=StoreBackend())</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Gives the agent access to long-term storage that is persisted across threads. Great for storing longer term memories or instructions that are applicable to the agent over multiple executions.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Context Hub<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=ContextHubBackend("my-agent"))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Stores files durably in a LangSmith Hub repo, without provisioning a separate LangGraph store.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Sandbox<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=sandbox)</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Execute code in isolated environments. Sandboxes provide filesystem tools plus the <code style="font-size:15px; color:#2b8a3e;">execute</code> tool for running shell commands. Choose from LangSmith, AgentCore, Daytona, Deno, E2B, Modal, Runloop, or local VFS.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Local shell<br><code style="font-size:15px; color:#2b8a3e;">create_deep_agent(model="google_genai:gemini-3.5-flash", backend=LocalShellBackend(root_dir=".", env={"PATH": "/usr/bin:/bin"}))</code></td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Filesystem and shell execution directly on the host. No isolation—use only in controlled development environments. See security considerations below.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Composite</td>
<td style="border:1px solid #999; padding:12px; text-align:left; vertical-align:top;">Thread-scoped by default, <code style="font-size:15px; color:#2b8a3e;">/memories/</code> persisted across threads. The Composite backend is maximally flexible. You can specify different routes in the filesystem to point towards different backends.</td>
</tr>
</tbody>
</table>

# Default Backend
- Default backend is thread-scoped
- Without a `thread_id` the agent cannot find the referenced filesystem
- Stored in-memory, or SQLLite, Postgres, Redis, MongoDB

####  Thread ID is how we tell our agent that our response is part of a larger conversation.

In [31]:
from langgraph.checkpoint.memory import MemorySaver

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, checkpointer=MemorySaver(),  # We don't have to mention the backend when using the 'default'; it is just available to us
)

config = {"configurable": {"thread_id": "joke-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Tell me a joke about computer scientists and write it to /joke.md"}]
}, config=config)
pretty_print(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Tool

Updated file /joke.md

### Ai

Wrote the joke to `/joke.md`.

#### We are able to read the filesystem because we pass in the same `config` object has the `thread_id` we previously used.

In [32]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=config)
print_exchange(result)

### Human

Tell me a joke about computer scientists and write it to /joke.md

### Ai

Why do computer scientists confuse Halloween and Christmas?

Because OCT 31 == DEC 25.

#### This time the agent cannot find the file because we use a different `thread_id`

In [33]:
other_config = {"configurable": {"thread_id": "other-session"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /joke.md and tell me the joke"}]
}, config=other_config)
print_exchange(result)

### Human

Read /joke.md and tell me the joke

### Ai

I couldn’t read `/joke.md` because the file wasn’t found at that path.

# Store Backend
- Long-term
- Backed by a database
- Persisted across threads
- Good for storing longer term memories or instructions that are applicable to the agent over multiple executions.

#### In this example we write a memory in a thread named 'session-a'

In [36]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=store,
)

# Write a memory in one thread...
session_a_config = {"configurable": {"thread_id": "session-a"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Remember that my name is Faisal by writing it to /profile.md"}]
}, config=session_a_config)

pretty_print(result)

### Human

Remember that my name is Faisal by writing it to /profile.md

### Tool

Updated file /profile.md

### Ai

Done.

#### Now we can read the memory back from a _different_ thread. Unlike the default backend, the **StoreBackend** persists files across threads.

In [37]:
session_b_config = {"configurable": {"thread_id": "session-b"}}

result = agent.invoke({
    "messages": [{"role": "user", "content": "Read /profile.md and tell me my name"}]
}, config=session_b_config)

print_exchange(result)

### Human

Read /profile.md and tell me my name

### Ai

Your name is Faisal.

In [39]:
# Per-user long-term memory: shared across a user's threads, isolated from every other user.
from dataclasses import dataclass
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore

@dataclass(frozen=True)
class UserContext:
    user_id: str  # In production, populate this from authenticated server-side identity.

def user_memory_namespace(runtime):
    if runtime.context is None or not runtime.context.user_id:
        raise PermissionError("Authenticated user required")
    return ("deepagents-basic", "v1", "users", runtime.context.user_id, "memories")

# InMemoryStore keeps this demo self-contained; use a database-backed BaseStore in production.
user_store = InMemoryStore()
user_memory_backend = CompositeBackend(
    default=StateBackend(),  # Scratch files remain scoped to each thread.
    routes={"/memories/": StoreBackend(namespace=user_memory_namespace)},
)

user_scoped_agent = create_deep_agent(
    model=model,
    system_prompt=research_system_prompt,
    backend=user_memory_backend,
    store=user_store,
    context_schema=UserContext,
    checkpointer=MemorySaver(),
)

def invoke_as(user_id: str, thread_id: str, message: str):
    return user_scoped_agent.invoke(
        {"messages": [{"role": "user", "content": message}]},
        config={"configurable": {"thread_id": thread_id}},
        context=UserContext(user_id=user_id),
    )

# User A writes in one thread.
invoke_as("user-a", "user-a-thread-1", "Remember that my name is Faisal by writing it to /memories/profile.md")

# User A can read the same file in a new thread.
same_user = invoke_as(
    "user-a", "user-a-thread-2",
    "Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.",
)
print("Same user, different thread:")
print_exchange(same_user)

# User B asks for the identical path but receives an isolated namespace.
different_user = invoke_as(
    "user-b", "user-b-thread-1",
    "Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.",
)
print("\nDifferent user:")
print_exchange(different_user)


Same user, different thread:


### Human

Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.

### Ai

Faisal


Different user:


### Human

Read /memories/profile.md. If it exists, tell me the name; otherwise say 'No memory found'.

### Ai

No memory found

# Store Backend: Choosing the backing database

The <code style="font-size:15px; color:#2b8a3e;">StoreBackend</code> is <b>storage-agnostic</b>. It is a thin filesystem adapter over whatever LangGraph <code style="font-size:15px; color:#2b8a3e;">BaseStore</code> you pass to <code style="font-size:15px; color:#2b8a3e;">create_deep_agent(store=...)</code>. The database is chosen entirely by that one argument &mdash; the agent code never changes.

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code>agent = create_deep_agent(
    model=model,
    backend=StoreBackend(namespace=lambda ctx: ("memories",)),
    store=<span style="color:#cf222e;">&lt;ANY BaseStore&gt;</span>,   <span style="color:#6e7781;"># &larr; this is the database choice</span>
)</code></pre>

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left;">Store</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Package</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Import</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">InMemoryStore</code> <span style="color:#6e7781;">(dev only)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left;">built-in</td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.memory</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">PostgresStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-postgres</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.postgres</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">RedisStore</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph-checkpoint-redis</code></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><code style="color:#2b8a3e;">langgraph.store.redis</code></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">MongoDBStore</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph-store-mongodb</code></b></td>
<td style="border:1px solid #999; padding:10px; text-align:left;"><b><code style="color:#2b8a3e;">langgraph.store.mongodb</code></b></td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left;">Anything custom</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">your code</td>
<td style="border:1px solid #999; padding:10px; text-align:left;">subclass <code style="color:#2b8a3e;">BaseStore</code></td>
</tr>
</tbody>
</table>

### MongoDB example

<pre style="background:#f6f8fa; border:1px solid #d0d7de; border-radius:6px; padding:12px; font-size:14px; line-height:1.5; overflow:auto;"><code><span style="color:#cf222e;">from</span> langgraph.store.mongodb <span style="color:#cf222e;">import</span> MongoDBStore

<span style="color:#cf222e;">with</span> MongoDBStore.from_conn_string(
    conn_string=<span style="color:#0a3069;">"mongodb://localhost:27017"</span>,
    db_name=<span style="color:#0a3069;">"deepagents"</span>,
    collection_name=<span style="color:#0a3069;">"memories"</span>,
) <span style="color:#cf222e;">as</span> store:
    agent = create_deep_agent(
        model=model,
        system_prompt=research_system_prompt,
        backend=StoreBackend(namespace=<span style="color:#cf222e;">lambda</span> ctx: (<span style="color:#0a3069;">"memories"</span>,)),
        store=store,   <span style="color:#6e7781;"># only this line changed &mdash; files now persist in MongoDB</span>
    )
    <span style="color:#6e7781;"># ...same invoke() calls as the InMemoryStore cell</span></code></pre>


# Composite Backend
- Flexible: specify different routes in the filesystem to point towards different backends.

#### Example:
- Use <code style="font-size:15px; color:#2b8a3e;">StateBackend</code> for thread-scoped default storage
- Save persistent memories to <code style="font-size:15px; color:#2b8a3e;">/memories/</code>. These files are available from different threads.
- Save ephemeral memories to the root <code style="font-size:15px; color:#2b8a3e;">/</code>. These are not available across different threads.



In [40]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

# Route by path: everything is ephemeral (StateBackend) except /memories/, which is routed to the durable StoreBackend.
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend(namespace=lambda ctx: ("memories",))},
)

agent = create_deep_agent(
    model=model, system_prompt=research_system_prompt, backend=composite_backend, store=store,
)

# Thread A: write one ephemeral file and one durable file
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Write a quick scratch note to /draft.md, and save that my name is Faisal to /memories/profile.md"
    )}]
}, config={"configurable": {"thread_id": "thread-a"}})
pretty_print(result)

### Human

Write a quick scratch note to /draft.md, and save that my name is Andrew to /memories/profile.md

### Tool

['/memories/']

### Tool

Updated file /draft.md

### Tool

Updated file /memories/profile.md

### Ai

Done.

In [41]:
# Thread B: a *different* thread. The /memories/ route persisted; /draft.md did not.
result = agent.invoke({
    "messages": [{"role": "user", "content": (
        "Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each."
    )}]
}, config={"configurable": {"thread_id": "thread-b"}})
print_exchange(result)

### Human

Try to read both /memories/profile.md and /draft.md. Report exactly what you find for each.

### Ai

`/memories/profile.md`:

```md
# Profile

Name: Andrew
```

`/draft.md`:

File not found.

# Context Isolation vs. Shared Filesystems

When a supervisor delegates to a subagent, two different things happen to two different kinds of state:

<table style="font-size:16px; border-collapse:collapse; width:100%;">
<thead>
<tr>
<th style="border:1px solid #999; padding:10px; text-align:left; width:22%;">What</th>
<th style="border:1px solid #999; padding:10px; text-align:left;">Behavior</th>
</tr>
</thead>
<tbody>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Message context</b><br><span style="color:#6e7781;">(isolated)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The subagent runs in its <b>own context window</b>. Its intermediate reasoning and tool calls never enter the supervisor's history &mdash; only a single, compact <b>final report</b> is returned. This is the whole point of subagents: keep the supervisor's context clean.</td>
</tr>
<tr>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;"><b>Filesystem</b><br><span style="color:#6e7781;">(shared)</span></td>
<td style="border:1px solid #999; padding:10px; text-align:left; vertical-align:top;">The backend is <b>shared</b> between the supervisor and all subagents. Any file a subagent writes <b>remains in state after the subagent finishes</b> and stays readable by the supervisor and sibling subagents.</td>
</tr>
</tbody>
</table>

### The pattern this unlocks

The filesystem is a **shared side-channel** for moving large results across the subagent boundary *without* paying for them in context:

1. Supervisor delegates: *"research X and write the findings to <code style="color:#2b8a3e;">/research.md</code>"*
2. Subagent does the heavy work, writes the file, and returns just *"wrote results to /research.md"*
3. Supervisor (or another subagent) reads <code style="color:#2b8a3e;">/research.md</code> when it needs the detail

The bulky output travels through the filesystem; only a one-line handoff travels through the conversation.

<b>Caveat:</b> dictionary subagents inherit the supervisor's backend automatically. A fully custom <code style="font-size:15px; color:#2b8a3e;">CompiledSubAgent</code> graph has its own state and only shares the filesystem if you wire it to the same backend.

In [25]:
from langgraph.checkpoint.memory import MemorySaver

# A subagent whose job is to write its findings to a file (not to report them inline).
writer_subagent = {
    "name": "writer-agent",
    "description": "Writes the requested content to a file, then reports only the path.",
    "system_prompt": (
        "Write the requested content to the requested file path using write_file. "
        "Then report ONLY the path you wrote to — do not repeat the content."
    ),
    "tools": [],
}

agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a supervisor. Delegate writing tasks to the writer-agent subagent via the task tool. "
        "The subagent shares your filesystem, so after it finishes you can read the file it wrote yourself."
    ),
    subagents=[writer_subagent],
    checkpointer=MemorySaver(),
)

config = {"configurable": {"thread_id": "subagent-fs-demo"}}

# print_activity streams the run with subgraphs=True, so we can watch the *subagent*
# call write_file inside its own isolated context — that tool call never appears in
# the supervisor's returned messages.
print_activity(agent, (
    "Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. "
    "After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me."
), config=config)

### Activity timeline

- 🧑‍✈️ **supervisor** → 📨 delegates via **task** to `writer-agent`
    - 🤖 **writer-agent** → 🔧 **write_file**(`/poem.md`, content=“Tides breathe under moonlight Salt wind combs the sleeping waves Blue heart hums…”)
    - 🤖 **writer-agent** ← 📥 `write_file` → Updated file /poem.md
- 🧑‍✈️ **supervisor** ← 📥 `task` → /poem.md
- 🧑‍✈️ **supervisor** → 🔧 **read_file**(`/poem.md`)
- 🧑‍✈️ **supervisor** ← 📥 `read_file` → 1 Tides breathe under moonlight 2 Salt wind combs the sleeping waves 3 Blue hear…

#### Because we passed a checkpointer + thread_id, the run's state is durable on this thread. The subagent's file lives in the shared filesystem, readable by the main supervisor agent.

In [26]:
state = agent.get_state(config)
print("Files in shared state:", list(state.values.get("files", {}).keys()))

# And the supervisor's own transcript only ever saw the delegation + the final read —
# never the subagent's write_file call (that stayed isolated in the subagent's context).
pretty_print(state.values)

Files in shared state: ['/poem.md']


### Human

Delegate to writer-agent: have it write a haiku about the ocean to /poem.md. After the subagent finishes, YOU read /poem.md with read_file and quote its exact contents back to me.

### Tool

/poem.md

### Tool

1  Tides breathe under moonlight
2  Salt wind combs the sleeping waves
3  Blue heart hums softly

### Ai

Here are the exact contents of `/poem.md`:

```text
Tides breathe under moonlight
Salt wind combs the sleeping waves
Blue heart hums softly
```

# Context Management Techniques

### A. Subagent Context Isolation

Each subagent gets its own, isolated context window. This ensures that context relevant to a specific task is only seen by the subagent working on that task.

### B. Offloading Large Inputs and Outputs

When a tool returns (or receives) more than comfortably fits in context, the harness writes the payload to the filesystem and replaces it in the message history with a short **reference plus a preview** (e.g. the first 10 lines). The agent calls <code style="color:#2b8a3e;">read_file</code> to pull back more **only if it actually needs the detail** — instead of dragging thousands of lines through every subsequent turn.

<img src="images/offloading-results.png" alt="Offloading large tool results to the filesystem" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

<ul style="font-size:16px;">
<li>Oversized tool results are evicted and swapped for a reference.</li>
<li>The agent reads back just the slice it needs via <code style="color:#2b8a3e;">read_file</code>.</li>
</ul>

### C. Context Summarization

When the **conversation itself** approaches the model's limit (~85% of <code style="color:#2b8a3e;">max_input_tokens</code>), <code style="color:#2b8a3e;">SummarizationMiddleware</code> runs a **compaction step**: older turns are compressed into an LLM-generated summary — session intent, artifacts created, next steps — while the most recent ~10% of tokens are kept verbatim.

<ul style="font-size:16px;">
<li><b>In-context summary + pointer:</b> the summary replaces the old messages and tells the agent where the full record lives.</li>
<li><b>Filesystem preservation:</b> the original messages are written to <code style="color:#2b8a3e;">/conversation_history/{thread_id}.md</code>, so the agent can search or re-read them if needed.</li>
</ul>

<img src="images/summarization.png" alt="Summarizing older conversation turns into a compact summary + pointer" style="max-width:100%; width:820px; border:1px solid #d0d7de; border-radius:6px;" />

